# Questão 3 — Árvore de decisão em uma base do Kaggle/UCI

Base escolhida: **Car Evaluation** (UCI Machine Learning Repository, também disponível
no Kaggle). O domínio é avaliação de carros: a partir de 6 características do veículo,
o modelo prevê se o carro é `unacc`, `acc`, `good` ou `vgood`.

Todas as colunas já são categóricas, o que combina bem com as árvores implementadas na
Questão 1 (ID3, C4.5, CART), que trabalham com atributos discretos. As árvores são
reaproveitadas de `Questao_1/Algoritimos`, igual foi feito na Questão 2.

## 1. Base de dados

`buying`, `maint`, `doors`, `persons`, `lug_boot`, `safety` são os atributos e `class`
é o alvo. Não tem valor faltando nem coluna de índice pra remover, então o tratamento
aqui é só nomear as colunas (o arquivo original não vem com cabeçalho).

In [1]:
import sys
import os
import pandas as pd

DATA_PATH = os.path.join("dados", "car_evaluation.csv")
df = pd.read_csv(DATA_PATH)
df.head()

,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


In [2]:
df["class"].value_counts()

class
unacc    1210
acc       384
good       69
vgood      65
Name: count, dtype: int64

## 2. Importando as árvores da Questão 1

In [3]:
ALGORITHMS_FOLDER = os.path.join("..", "Questao_1", "Algoritimos")
sys.path.append(ALGORITHMS_FOLDER)

from id3 import id3
from c45 import c45
from cart import cart

## 3. Separando treino e teste

Diferente das questões anteriores (onde a árvore era avaliada na própria base de
treino), aqui o enunciado pede pra avaliar com acurácia, F1, precisão e recall — então
faz sentido separar uma parte da base só pra teste (20%), estratificada pela classe.

In [4]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2, stratify=df["class"], random_state=42)
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

len(train), len(test)

(1382, 346)

## 4. Extração das regras

As árvores guardam a estrutura em objetos `Node`. Pra virar uma base de regras (tipo
"SE buying=vhigh E safety=low ENTÃO class=unacc"), basta percorrer a árvore até cada
folha guardando as condições do caminho.

In [5]:
def extract_rules(node, path=None):
    if path is None:
        path = []

    if node.is_leaf():
        return [f"SE {' E '.join(path)} ENTAO class = {node.label}"]

    rules = []
    for value, child in node.children.items():
        rules += extract_rules(child, path + [f"{node.attribute} = {value}"])
    return rules

## 5. Métricas de avaliação

Uma função só pra não repetir o cálculo de acurácia, precisão, recall e F1 três vezes
(usa `average="weighted"` porque a classe é multiclasse e desbalanceada).

In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

def evaluate(predictions):
    real, predicted = predictions["Real"], predictions["Predito"]
    return {
        "acuracia": accuracy_score(real, predicted),
        "precisao": precision_score(real, predicted, average="weighted", zero_division=0),
        "recall": recall_score(real, predicted, average="weighted", zero_division=0),
        "f1": f1_score(real, predicted, average="weighted", zero_division=0),
    }

## 6. ID3

In [7]:
id3_tree = id3(train, verbose=True)
id3_tree.fit("class")

id3_tree.export_tree(os.path.join("outputs", "id3_log.txt"))
id3_tree.export_pure_tree(os.path.join("outputs", "id3_arvore.txt"))

id3_rules = extract_rules(id3_tree.root)
with open(os.path.join("outputs", "id3_regras.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(id3_rules))

print(f"{len(id3_rules)} regras geradas, algumas:")
for r in id3_rules[:5]:
    print(r)

252 regras geradas, algumas:
SE safety = med E persons = 4 E buying = med E maint = med ENTAO class = acc
SE safety = med E persons = 4 E buying = med E maint = low E lug_boot = med E doors = 5more ENTAO class = good
SE safety = med E persons = 4 E buying = med E maint = low E lug_boot = med E doors = 2 ENTAO class = acc
SE safety = med E persons = 4 E buying = med E maint = low E lug_boot = med E doors = 3 ENTAO class = acc
SE safety = med E persons = 4 E buying = med E maint = low E lug_boot = med E doors = 4 ENTAO class = good


In [8]:
id3_predictions = id3_tree.classify_dataset(df=test, filepath=os.path.join("outputs", "id3_predicoes.csv"))
id3_metrics = evaluate(id3_predictions)
id3_metrics

{'acuracia': 0.9075144508670521,
 'precisao': 0.9121047262835771,
 'recall': 0.9075144508670521,
 'f1': 0.9017293106910604}

## 7. C4.5

In [9]:
c45_tree = c45(train, verbose=True)
c45_tree.fit("class")

c45_tree.export_tree(os.path.join("outputs", "c45_log.txt"))
c45_tree.export_pure_tree(os.path.join("outputs", "c45_arvore.txt"))

c45_rules = extract_rules(c45_tree.root)
with open(os.path.join("outputs", "c45_regras.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(c45_rules))

len(c45_rules)

246

In [10]:
c45_predictions = c45_tree.classify_dataset(df=test, filepath=os.path.join("outputs", "c45_predicoes.csv"))
c45_metrics = evaluate(c45_predictions)
c45_metrics

{'acuracia': 0.8959537572254336,
 'precisao': 0.8963219759942144,
 'recall': 0.8959537572254336,
 'f1': 0.8910283679069807}

## 8. CART

In [11]:
cart_tree = cart(train, verbose=True)
cart_tree.fit("class")

cart_tree.export_tree(os.path.join("outputs", "cart_log.txt"))
cart_tree.export_pure_tree(os.path.join("outputs", "cart_arvore.txt"))

cart_rules = extract_rules(cart_tree.root)
with open(os.path.join("outputs", "cart_regras.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(cart_rules))

len(cart_rules)

253

In [12]:
cart_predictions = cart_tree.classify_dataset(df=test, filepath=os.path.join("outputs", "cart_predicoes.csv"))
cart_metrics = evaluate(cart_predictions)
cart_metrics

{'acuracia': 0.9046242774566474,
 'precisao': 0.9083848873142739,
 'recall': 0.9046242774566474,
 'f1': 0.8991624263860637}

## 9. Comparando os três

In [13]:
comparison = pd.DataFrame({
    "ID3": id3_metrics,
    "C4.5": c45_metrics,
    "CART": cart_metrics,
}).T
comparison

,acuracia,precisao,recall,f1
ID3,0.907514,0.912105,0.907514,0.901729
C4.5,0.895954,0.896322,0.895954,0.891028
CART,0.904624,0.908385,0.904624,0.899162


Os três ficam bem próximos (a base é pequena e totalmente categórica, então a escolha
da métrica de ganho influencia pouco). O relatório detalhado por classe abaixo é do ID3,
só pra ilustrar onde o modelo erra mais (normalmente nas classes minoritárias `good` e
`vgood`, que têm poucos exemplos de treino).

In [14]:
print(classification_report(id3_predictions["Real"], id3_predictions["Predito"], zero_division=0))

              precision    recall  f1-score   support

         acc       0.93      0.82      0.87        77
        good       1.00      0.57      0.73        14
       unacc       0.90      0.98      0.94       242
       vgood       1.00      0.46      0.63        13

    accuracy                           0.91       346
   macro avg       0.96      0.71      0.79       346
weighted avg       0.91      0.91      0.90       346



## 10. Arquivos gerados

Tudo em `Questao_3/outputs/`: `<algoritmo>_log.txt` (log de construção), `<algoritmo>_arvore.txt`
(árvore em texto), `<algoritmo>_regras.txt` (base de regras) e `<algoritmo>_predicoes.csv`
(predições no conjunto de teste, usadas para calcular as métricas acima).